# 01. Tổng quan dữ liệu Abalone

Nội dung chính:
- Xác định mục tiêu bài toán.
- Nạp và mô tả bộ dữ liệu Abalone.
- Kiểm tra kích thước, kiểu dữ liệu, missing value, duplicate.
- Xem nhanh biến mục tiêu `Rings` và biến phân loại `Sex`.
- Rút ra các nhận xét sơ bộ cho bước tiền xử lý.


In [ ]:
import sys
import io
from pathlib import Path

current_path = Path.cwd().resolve()
candidate_paths = [current_path, *current_path.parents]
PROJECT_ROOT = next(
    (
        path for path in candidate_paths
        if (path / 'data').exists() and (path / 'src').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Không thể tự động xác định thư mục dự án AbaloneAge. '
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.data.load_data import ABALONE_COLUMNS, load_abalone_data

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Mục tiêu bài toán


In [2]:
problem_summary = pd.DataFrame(
    {
        'nội_dung': [
            'Loại bài toán',
            'Biến mục tiêu',
            'Ý nghĩa của Rings',
            'Mục tiêu EDA content 1',
        ],
        'giá_trị': [
            'Hồi quy',
            'Rings',
            'Số vòng tăng trưởng của abalone, thường được dùng để suy ra tuổi gần đúng',
            'Tổng quan bộ dữ liệu, chất lượng dữ liệu, và định hướng cho các notebook EDA sau',
        ],
    }
)
display(problem_summary)


,nội_dung,giá_trị
0,Loại bài toán,Hồi quy
1,Biến mục tiêu,Rings
2,Ý nghĩa của Rings,"Số vòng tăng trưởng của abalone, thường được d..."
3,Mục tiêu EDA content 1,"Tổng quan bộ dữ liệu, chất lượng dữ liệu, và đ..."


## 2. Nạp dữ liệu gốc


In [3]:
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'
df = load_abalone_data(DATA_PATH, columns=ABALONE_COLUMNS.copy())

display_path = DATA_PATH.relative_to(PROJECT_ROOT) if DATA_PATH.is_relative_to(PROJECT_ROOT) else DATA_PATH
print(f'Dường dẫn dữ liệu: {display_path}')
display(df.head())


Dường dẫn dữ liệu: C:\Users\ACER\Documents\GitHub\TNTT_Repo_Abalone6\AbaloneAge\data\raw\abalone.csv


,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 3. Từ điển dữ liệu


In [4]:
data_dictionary = pd.DataFrame(
    {
        'cột': ABALONE_COLUMNS,
        'mô_tả': [
            'Giới tính của abalone: M, F, I',
            'Chiều dài lớn nhất của vỏ',
            'Đường kính vuông góc với chiều dài',
            'Chiều cao của abalone trong vỏ',
            'Khối lượng toàn bộ con abalone',
            'Khối lượng phần thịt đã tách',
            'Khối lượng nội tạng sau khi lấy mẫu',
            'Khối lượng vỏ sau khi sấy khô',
            'Số vòng tăng trưởng trên vỏ; biến mục tiêu cần dự đoán',
        ],
    }
)
display(data_dictionary)


,cột,mô_tả
0,Sex,"Giới tính của abalone: M, F, I"
1,Length,Chiều dài lớn nhất của vỏ
2,Diameter,Đường kính vuông góc với chiều dài
3,Height,Chiều cao của abalone trong vỏ
4,WholeWeight,Khối lượng toàn bộ con abalone
5,ShuckedWeight,Khối lượng phần thịt đã tách
6,VisceraWeight,Khối lượng nội tạng sau khi lấy mẫu
7,ShellWeight,Khối lượng vỏ sau khi sấy khô
8,Rings,Số vòng tăng trưởng trên vỏ; biến mục tiêu cần...


## 4. Kích thước và kiểu dữ liệu


In [5]:
print(f'Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột')
display(df.dtypes.to_frame('dtype'))

buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

display(df.describe(include='all').T)


Kích thước dữ liệu: 4177 dòng, 9 cột


,dtype
Sex,object
Length,float64
Diameter,float64
Height,float64
WholeWeight,float64
ShuckedWeight,float64
VisceraWeight,float64
ShellWeight,float64
Rings,int64


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sex            4177 non-null   object 
 1   Length         4177 non-null   float64
 2   Diameter       4177 non-null   float64
 3   Height         4177 non-null   float64
 4   WholeWeight    4177 non-null   float64
 5   ShuckedWeight  4177 non-null   float64
 6   VisceraWeight  4177 non-null   float64
 7   ShellWeight    4177 non-null   float64
 8   Rings          4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB



,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Sex,4177,3,M,1528,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Length,4177.0,NaN,NaN,NaN,0.523992,0.120093,0.075,0.45,0.545,0.615,0.815
Diameter,4177.0,NaN,NaN,NaN,0.407881,0.09924,0.055,0.35,0.425,0.48,0.65
Height,4177.0,NaN,NaN,NaN,0.139516,0.041827,0.0,0.115,0.14,0.165,1.13
WholeWeight,4177.0,NaN,NaN,NaN,0.828742,0.490389,0.002,0.4415,0.7995,1.153,2.8255
ShuckedWeight,4177.0,NaN,NaN,NaN,0.359367,0.221963,0.001,0.186,0.336,0.502,1.488
VisceraWeight,4177.0,NaN,NaN,NaN,0.180594,0.109614,0.0005,0.0935,0.171,0.253,0.76
ShellWeight,4177.0,NaN,NaN,NaN,0.238831,0.139203,0.0015,0.13,0.234,0.329,1.005
Rings,4177.0,NaN,NaN,NaN,9.933684,3.224169,1.0,8.0,9.0,11.0,29.0


## 5. Kiểm tra missing value và duplicate


In [6]:
missing_summary = df.isna().sum().to_frame('missing_count')
missing_summary['missing_ratio'] = (missing_summary['missing_count'] / len(df)).round(4)
duplicate_count = int(df.duplicated().sum())

display(missing_summary)
print(f'So dong trung lap: {duplicate_count}')


,missing_count,missing_ratio
Sex,0,0.0
Length,0,0.0
Diameter,0,0.0
Height,0,0.0
WholeWeight,0,0.0
ShuckedWeight,0,0.0
VisceraWeight,0,0.0
ShellWeight,0,0.0
Rings,0,0.0


So dong trung lap: 0


## 6. Tổng quan biến mục tiêu `Rings`


In [7]:
rings_summary = df['Rings'].describe().to_frame().T
rings_summary['skewness'] = df['Rings'].skew()
display(rings_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Rings'], kde=True, bins=25, ax=axes[0])
axes[0].set_title('Phan phoi cua Rings')
sns.boxplot(x=df['Rings'], ax=axes[1])
axes[1].set_title('Boxplot cua Rings')
plt.tight_layout()
plt.show()


,count,mean,std,min,25%,50%,75%,max,skewness
Rings,4177.0,9.933684,3.224169,1.0,8.0,9.0,11.0,29.0,1.114102


C:\Users\ACER\AppData\Local\Temp\ipykernel_5704\2267066657.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Tổng quan biến phân loại `Sex`


In [8]:
sex_counts = df['Sex'].value_counts().rename_axis('Sex').reset_index(name='count')
sex_target_summary = df.groupby('Sex')['Rings'].agg(['count', 'mean', 'median', 'std']).round(2)

display(sex_counts)
display(sex_target_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='Sex', order=df['Sex'].value_counts().index, ax=axes[0])
axes[0].set_title('Phân phối của Sex')
sns.boxplot(data=df, x='Sex', y='Rings', order=df['Sex'].value_counts().index, ax=axes[1])
axes[1].set_title('Phân phối Rings theo Sex')
plt.tight_layout()
plt.show()


,Sex,count
0,M,1528
1,I,1342
2,F,1307


,count,mean,median,std
Sex,,,,
F,1307,11.13,10.0,3.10
I,1342,7.89,8.0,2.51
M,1528,10.71,10.0,3.03


C:\Users\ACER\AppData\Local\Temp\ipykernel_5704\927415560.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Tổng hợp nhận xét sơ bộ


In [9]:
eda_notes = pd.DataFrame(
    {
        'nhận_xét': [
            'Bộ dữ liệu có cấu trúc gọn',
            'Không có missing value',
            'Không có duplicate đang kể',
            'Sex là biến phân loại cần mã hóa',
            'Rings có phân phối hơi lệch và có giá trị biến',
            'Lần tiếp tục kiểm tra outlier và tương quan ở các notebook sau',
        ],
        'hàm_ý_cho_bước_sau': [
            'Có thể thao tác trực tiếp và chia nhỏ theo cụm notebook research',
            'không cần xử lý dữ liệu khuyết trong preprocessing baseline',
            'Chỉ cần kiểm tra lại nhánh ở notebook preprocessing',
            'Cần one-hot encoding hoặc cách mã hóa phù hợp',
            'Lên đánh giá scaling và outlier handling',
            'Dùng 01_eda_outlier_check và 01_eda_correlation để điều chỉnh sau hơn',
        ],
    }
)
display(eda_notes)


,nhận_xét,hàm_ý_cho_bước_sau
0,Bộ dữ liệu có cấu trúc gọn,Có thể thao tác trực tiếp và chia nhỏ theo cụm...
1,Không có missing value,không cần xử lý dữ liệu khuyết trong preproces...
2,Không có duplicate đang kể,Chỉ cần kiểm tra lại nhánh ở notebook preproce...
3,Sex là biến phân loại cần mã hóa,Cần one-hot encoding hoặc cách mã hóa phù hợp
4,Rings có phân phối hơi lệch và có giá trị biến,Lên đánh giá scaling và outlier handling
5,Lần tiếp tục kiểm tra outlier và tương quan ở ...,Dùng 01_eda_outlier_check và 01_eda_correlatio...
